# AMOS 2D → 3D Pipeline

This notebook covers the full post-training pipeline for the AMOS multi-organ segmentation project:

| Step | Script | What it does |
|------|---------|--------------|
| 1 | `preprocessing.py` | Apply HU windowing / image enhancement to raw 2D slices before (re-)training |
| 2 | `evaluate_3d.py`   | Run a trained 2D model on raw NIfTI volumes slice-by-slice, reconstruct 3D predictions, compute DSC / HD95 / NSD |
| 3 | `postprocessing.py`| Clean up raw predicted 2D masks (CCA, morphological closing, 2.5D consistency) |

**Prerequisites**
- Dataset extracted with the AMOS extraction script (uint16 PNGs in `Data/Train|Test|Val/fold_1/`)
- At least one model trained with `TrainUnet.py` (checkpoint `.pt` in `Results/`)
- `config_2d3d.py` filled in with your paths (same directory as this notebook)

## 0 — Configuration

Edit `config_2d3d.py` before running. Key fields:

```python
config['data_root']        = r'C:/Salam/AMOS/2D-Seg/Data'        # extracted dataset root
config['nifti_images_dir'] = r'C:/Salam/AMOS/3D/amos/imagesTs'  # original NIfTI CTs
config['nifti_labels_dir'] = r'C:/Salam/AMOS/3D/amos/labelsTs'  # original NIfTI labels
config['model_path']       = r'C:/.../.../model_fold_1.pt'       # trained checkpoint

config['in_channels']      = 1      # must match training config.py
config['model_input_size'] = 256    # must match Resize_h in training config.py
config['input_mean']       = [0.2277]
config['input_std']        = [0.2317]

config['preprocess_method']  = 'grayscale'   # for 1-channel models
config['postprocess_method'] = 'cca_morph'
```

In [ ]:
import sys
import os

# Ensure this directory is on the path so imports work
NOTEBOOK_DIR = os.path.dirname(os.path.abspath('__file__'))
if NOTEBOOK_DIR not in sys.path:
    sys.path.insert(0, NOTEBOOK_DIR)

from importlib import import_module
cfg = import_module('config_2d3d').config

print("config_2d3d loaded successfully.")
print()
for k, v in cfg.items():
    print(f"  {k:<22} = {v}")

In [ ]:
# Common imports used throughout the notebook
import json
import numpy as np
import torch
import nibabel as nib
import matplotlib.pyplot as plt
from pathlib import Path
from PIL import Image
from tqdm import tqdm

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

---
## 1 — Preprocessing

`preprocessing.py` converts raw uint16 PNG slices into a new dataset copy where each image has been enhanced with an HU windowing method. This is useful when you want to experiment with different preprocessing strategies without re-running the AMOS extraction.

### How it works
- Reads each image from `data_root/Train|Test|Val/fold_1/images/`
- Decodes raw HU: `HU = stored_pixel − 1024`
- Applies the selected method (windowing, CLAHE, unsharp, gamma)
- Saves the result to a new folder: `fold_1_<method>/images/`
- Copies masks unchanged to `fold_1_<method>/masks/`

### Preprocessing methods

| Method | Channels | Description |
|--------|----------|-------------|
| `grayscale` | 1 | Simple linear normalization: `(HU+1024)/4095` — **use this for 1-channel models** |
| `single_window` | 3 | Organ window `(40, 400)` replicated 3× |
| `multi_window` | 3 | Soft-tissue `(-60,400)` / Organ `(40,400)` / Vessel `(200,700)` windows |
| `multi_window_clahe` | 3 | Multi-window + CLAHE on each channel |
| `multi_window_clahe_unsharp` | 3 | + Unsharp masking on the organ channel |
| `multi_window_gamma` | 3 | Multi-window + gamma correction (γ=0.8) |
| `multi_window_clahe_gamma_unsharp` | 3 | Full pipeline |

> **Note:** `in_channels` in `config_2d3d.py` must match the number of channels the selected method produces. If your model was trained with `in_channels=1`, use `grayscale`. If `in_channels=3`, use any of the multi-window methods.

### To use preprocessed images for training
After running this step, update `config.py` in the `2D_Segmentation` folder:
```python
config['in_channels'] = 3              # or 1 for grayscale
config['input_mean']  = [0.485, 0.456, 0.406]   # recompute with image_mean_std.py
config['input_std']   = [0.229, 0.224, 0.225]
# Then update parentdir to point to the new fold_1_<method> folder structure
```

In [ ]:
from preprocessing import METHODS, process_split

# List available methods
print("Available preprocessing methods:")
for name in METHODS:
    tag = "  ← configured" if name == cfg['preprocess_method'] else ""
    print(f"  {name}{tag}")

In [ ]:
# Run preprocessing on all splits
# Output: data_root/<Split>/fold_1_<method>/

method_name = cfg['preprocess_method']
data_root   = Path(cfg['data_root'])
fn          = METHODS[method_name]

print(f"Applying: {method_name}")
print(f"Data root: {data_root}")
print()

for subdir, split_name in cfg['splits']:
    process_split(subdir, split_name, method_name, fn, data_root)

print("\nPreprocessing complete.")
print(f"New folders created as: {data_root}/<Split>/fold_1_{method_name}/")

---
## 2 — 3D Evaluation

`evaluate_3d.py` loads a trained 2D model and runs it slice-by-slice on original 3D NIfTI CT volumes. The 2D predictions are stacked back into a 3D volume and compared against the ground truth using three volumetric metrics:

| Metric | Meaning |
|--------|---------|
| **DSC** | Dice Similarity Coefficient — overlap percentage (higher = better) |
| **HD95** | 95th-percentile Hausdorff Distance in mm — boundary error (lower = better) |
| **NSD** | Normalized Surface Distance — fraction of surface within 1 mm (higher = better) |

### Inference pipeline per volume
1. Load NIfTI CT + GT label
2. For each axial slice:
   - Crop body ROI (same algorithm as the extraction script)
   - Resize crop to 512×512 (intermediate)
   - Apply selected preprocessing method + resize to `model_input_size`
   - Run model → predicted class map
   - Resize prediction back to original crop shape and place in full slice
3. Compute DSC, HD95, NSD per organ class against GT
4. Save predicted NIfTI + metrics JSON to `eval_output_dir`

### Preprocessing method for evaluation
Must match what the model was **trained** with:
- If trained with raw uint16 PNGs + grayscale normalization → use `'grayscale'`
- If trained with a preprocessed dataset (e.g. `multi_window_clahe`) → use that same method

In [ ]:
from evaluate_3d import load_model, PREPROCESS_METHODS

# List available preprocessing methods for evaluation
print("Available evaluation preprocessing methods:")
for name in PREPROCESS_METHODS:
    tag = "  ← configured" if name == cfg['preprocess_method'] else ""
    print(f"  {name}{tag}")

In [ ]:
# Load the trained model
model = load_model(cfg['model_path'], device)

print(f"Model loaded from: {cfg['model_path']}")
print(f"in_channels      : {cfg['in_channels']}")
print(f"model_input_size : {cfg['model_input_size']}")
print(f"mean / std       : {cfg['input_mean']} / {cfg['input_std']}")

### 2a — Single Volume Evaluation

Evaluate on one NIfTI volume to verify everything is working before running the full batch.

In [ ]:
from evaluate_3d import infer_volume

# ── Set paths for one test volume ──────────────────────────────────────
image_path = r'C:/Salam/AMOS/3D/amos/imagesTs/amos_0001.nii.gz'  # ← edit
label_path = r'C:/Salam/AMOS/3D/amos/labelsTs/amos_0001.nii.gz'  # ← edit
# ───────────────────────────────────────────────────────────────────────

out_dir = Path(cfg['eval_output_dir'])

report = infer_volume(
    nifti_image_path  = image_path,
    nifti_label_path  = label_path,
    model             = model,
    device            = device,
    preprocess_method = cfg['preprocess_method'],
    out_dir           = out_dir,
    in_channels       = cfg['in_channels'],
    mean              = cfg['input_mean'],
    std               = cfg['input_std'],
    input_size        = cfg['model_input_size'],
)

print(f"\nMean DSC : {report['mean_dsc']:.2f}%")
print(f"Mean HD95: {report['mean_hd95']:.2f} mm")
print(f"Mean NSD : {report['mean_nsd']:.2f}%")

In [ ]:
# Show per-organ results for the evaluated volume
print(f"{'Organ':<25} {'DSC %':>8} {'HD95 mm':>10} {'NSD %':>8}")
print("-" * 55)
for organ, m in report['per_class'].items():
    hd  = f"{m['hd95']:>10.2f}" if m['hd95'] is not None else f"{'N/A':>10}"
    tag = "" if m['present'] else "  (absent in GT)"
    print(f"{organ:<25} {m['dsc']:>8.2f} {hd} {m['nsd']:>8.2f}{tag}")

In [ ]:
# Visualize: overlay prediction vs ground truth on a mid-axial slice
volume_stem = Path(image_path).name.replace('.nii.gz', '').replace('.nii', '')
pred_nii    = nib.load(str(out_dir / f"{volume_stem}_pred.nii.gz"))
gt_nii      = nib.load(label_path)
ct_nii      = nib.load(image_path)

pred_vol = pred_nii.get_fdata(dtype=np.float32)
gt_vol   = gt_nii.get_fdata(dtype=np.float32)
ct_vol   = ct_nii.get_fdata(dtype=np.float32)

D        = ct_vol.shape[2]
z        = D // 2  # mid slice

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
ct_disp   = np.clip(ct_vol[:, :, z], -200, 400)

axes[0].imshow(ct_disp.T, cmap='gray', origin='lower')
axes[0].set_title(f'CT (z={z})')

axes[1].imshow(ct_disp.T, cmap='gray', origin='lower')
axes[1].imshow(gt_vol[:, :, z].T, cmap='nipy_spectral', alpha=0.5, vmin=0, vmax=15, origin='lower')
axes[1].set_title('Ground Truth')

axes[2].imshow(ct_disp.T, cmap='gray', origin='lower')
axes[2].imshow(pred_vol[:, :, z].T, cmap='nipy_spectral', alpha=0.5, vmin=0, vmax=15, origin='lower')
axes[2].set_title('Prediction')

for ax in axes:
    ax.axis('off')
plt.tight_layout()
plt.show()

### 2b — Batch Evaluation (All Test Volumes)

Run evaluation on every NIfTI volume found in `nifti_images_dir`. Results are saved per-volume as JSON files in `eval_output_dir`, then aggregated into a summary table.

In [ ]:
import pandas as pd
from evaluate_3d import infer_volume, ORGAN_MAP

img_dir = Path(cfg['nifti_images_dir'])
lbl_dir = Path(cfg['nifti_labels_dir'])
out_dir = Path(cfg['eval_output_dir'])

img_files = sorted(list(img_dir.glob('*.nii.gz')) + list(img_dir.glob('*.nii')))
print(f"Found {len(img_files)} volumes in {img_dir}")

all_reports = []

for img_path in img_files:
    stem     = img_path.name.replace('.nii.gz', '').replace('.nii', '')
    lbl_path = None
    for ext in ['.nii.gz', '.nii']:
        candidate = lbl_dir / (stem + ext)
        if candidate.exists():
            lbl_path = candidate
            break

    if lbl_path is None:
        print(f"  [skip] No label found for {stem}")
        continue

    # Skip if already evaluated
    json_out = out_dir / f"{stem}_metrics.json"
    if json_out.exists():
        print(f"  [cached] {stem}")
        with open(json_out) as f:
            all_reports.append(json.load(f))
        continue

    print(f"  Evaluating: {stem}")
    report = infer_volume(
        nifti_image_path  = str(img_path),
        nifti_label_path  = str(lbl_path),
        model             = model,
        device            = device,
        preprocess_method = cfg['preprocess_method'],
        out_dir           = out_dir,
        in_channels       = cfg['in_channels'],
        mean              = cfg['input_mean'],
        std               = cfg['input_std'],
        input_size        = cfg['model_input_size'],
    )
    all_reports.append(report)

print(f"\nEvaluated {len(all_reports)} volumes.")

In [ ]:
# Aggregate per-organ metrics across all volumes
organ_names = [ORGAN_MAP[c] for c in range(1, 16)]
rows = []

for report in all_reports:
    row = {'volume': report['volume']}
    for c in range(1, 16):
        name = ORGAN_MAP[c]
        m    = report['per_class'].get(name, {})
        row[f"{name}_dsc"]  = m.get('dsc',  None)
        row[f"{name}_hd95"] = m.get('hd95', None)
        row[f"{name}_nsd"]  = m.get('nsd',  None)
    row['mean_dsc']  = report['mean_dsc']
    row['mean_hd95'] = report['mean_hd95']
    row['mean_nsd']  = report['mean_nsd']
    rows.append(row)

df = pd.DataFrame(rows)

# Summary across volumes
print("\n── Mean metrics across all evaluated volumes ──")
print(f"  Mean DSC : {df['mean_dsc'].mean():.2f}%")
print(f"  Mean HD95: {df['mean_hd95'].mean():.2f} mm")
print(f"  Mean NSD : {df['mean_nsd'].mean():.2f}%")

# Per-organ summary
print(f"\n{'Organ':<25} {'DSC%':>8} {'HD95mm':>10} {'NSD%':>8}")
print("-" * 55)
for name in organ_names:
    dsc  = df[f"{name}_dsc"].dropna().mean()
    hd95 = df[f"{name}_hd95"].dropna().mean()
    nsd  = df[f"{name}_nsd"].dropna().mean()
    print(f"{name:<25} {dsc:>8.2f} {hd95:>10.2f} {nsd:>8.2f}")

# Save summary CSV
summary_path = out_dir / 'summary_metrics.csv'
df.to_csv(summary_path, index=False)
print(f"\nSaved summary CSV → {summary_path}")

In [ ]:
# Bar chart: mean DSC per organ across all volumes
dsc_means = [df[f"{name}_dsc"].dropna().mean() for name in organ_names]

fig, ax = plt.subplots(figsize=(14, 5))
bars = ax.bar(organ_names, dsc_means, color='steelblue', edgecolor='white')
ax.set_ylabel('DSC (%)')
ax.set_title('Mean 3D DSC per Organ')
ax.set_ylim(0, 100)
ax.axhline(np.nanmean(dsc_means), color='red', linestyle='--', label=f"Overall mean: {np.nanmean(dsc_means):.1f}%")
ax.legend()
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

---
## 3 — Postprocessing

`postprocessing.py` cleans up raw predicted 2D masks from `TrainUnet.py` (the PNG files saved to `Generated_mask/`).

### Methods

| Method | What it does | When to use |
|--------|-------------|-------------|
| `cca` | Keep only the **largest connected component** per organ class — removes isolated false-positive blobs | Always useful |
| `morphological` | **Binary closing** per class — fills small holes in predictions | When predictions have gaps |
| `cca_morph` | CCA then morphological closing — **recommended default** | Best general-purpose |
| `slice_consistency` | **2.5D mode filter** — smooths predictions across adjacent slices | When z-axis is noisy |
| `full` | CCA + morphological closing + slice consistency | Most aggressive cleaning |

### Input format
Expects PNG files named `{volume_stem}_z{zzzz}.png` (e.g., `amos_0001_z0042.png`) — this matches the naming convention from the AMOS extraction pipeline.

### Output
New directory with the same filenames, postprocessed masks preserved as uint8 PNG with values 0–15.

In [ ]:
from postprocessing import METHODS as POST_METHODS, process_directory

# List available postprocessing methods
print("Available postprocessing methods:")
for name in POST_METHODS:
    tag = "  ← configured" if name == cfg['postprocess_method'] else ""
    print(f"  {name}{tag}")

In [ ]:
# Run postprocessing
method_name = cfg['postprocess_method']
pred_dir    = Path(cfg['pred_dir'])
post_dir    = Path(cfg['post_output_dir'])

print(f"Method     : {method_name}")
print(f"Input dir  : {pred_dir}")
print(f"Output dir : {post_dir}")
print()

fn = POST_METHODS[method_name]
process_directory(pred_dir, post_dir, method_name, fn)

print("\nPostprocessing complete.")

In [ ]:
# Compare a raw vs postprocessed mask side by side
# Pick any PNG from the prediction directory to visualise
sample_files = sorted(pred_dir.glob('*.png'))
if not sample_files:
    print("No prediction files found in pred_dir. Run TrainUnet.py first.")
else:
    sample_name = sample_files[len(sample_files) // 2].name  # pick a mid-range slice

    raw_mask  = np.array(Image.open(pred_dir  / sample_name).convert('L'))
    post_mask = np.array(Image.open(post_dir  / sample_name).convert('L'))

    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    axes[0].imshow(raw_mask,  cmap='nipy_spectral', vmin=0, vmax=15)
    axes[0].set_title(f'Raw prediction\n{sample_name}')
    axes[1].imshow(post_mask, cmap='nipy_spectral', vmin=0, vmax=15)
    axes[1].set_title(f'After "{method_name}"\n{sample_name}')
    for ax in axes:
        ax.axis('off')
    plt.tight_layout()
    plt.show()

---
## 4 — Combined Results Summary

Load all saved JSON metric files from `eval_output_dir` and produce a final report.

In [ ]:
from evaluate_3d import ORGAN_MAP

out_dir     = Path(cfg['eval_output_dir'])
json_files  = sorted(out_dir.glob('*_metrics.json'))

if not json_files:
    print("No metric files found. Run Section 2 first.")
else:
    all_reports = [json.load(open(f)) for f in json_files]
    organ_names = [ORGAN_MAP[c] for c in range(1, 16)]

    print(f"Loaded {len(all_reports)} volume reports from {out_dir}")
    print()

    # Mean across volumes per organ
    print(f"{'Organ':<25} {'DSC%':>8} {'HD95mm':>10} {'NSD%':>8}")
    print("-" * 55)

    all_dsc = []
    for name in organ_names:
        vals_dsc  = [r['per_class'][name]['dsc']  for r in all_reports if name in r['per_class'] and r['per_class'][name]['present']]
        vals_hd95 = [r['per_class'][name]['hd95'] for r in all_reports if name in r['per_class'] and r['per_class'][name]['present'] and r['per_class'][name]['hd95'] is not None]
        vals_nsd  = [r['per_class'][name]['nsd']  for r in all_reports if name in r['per_class'] and r['per_class'][name]['present']]

        dsc  = np.mean(vals_dsc)  if vals_dsc  else float('nan')
        hd95 = np.mean(vals_hd95) if vals_hd95 else float('nan')
        nsd  = np.mean(vals_nsd)  if vals_nsd  else float('nan')
        all_dsc.append(dsc)

        print(f"{name:<25} {dsc:>8.2f} {hd95:>10.2f} {nsd:>8.2f}")

    print("-" * 55)
    print(f"{'Overall mean':<25} {np.nanmean(all_dsc):>8.2f}")

    # Timing summary
    times = [r['inference_time_s'] for r in all_reports]
    print(f"\nInference time: {np.mean(times):.1f}s ± {np.std(times):.1f}s per volume")